In [1]:
import json, math
from datetime import datetime
from pathlib import Path
from sentence_transformers import SentenceTransformer

NODES = Path("nodes.json")
EMBS  = Path("embeddings.json")

MODEL = "all-MiniLM-L6-v2"
TOP_K = 10

# paper-ish knobs
HALF_LIFE_HOURS = 24.0
W_RECENCY = 0.15
W_IMPORTANCE = 0.80
W_RELEVANCE = 2.50

REL_MIN = 0.62  # IMPORTANT: this removes bed/idle spam

TYPE_W = {"thought": 2.0, "chat": 1.7, "event": 1.0}

def parse_dt(s):
    for fmt in ("%Y-%m-%d %H:%M:%S", "%Y-%m-%d %H:%M"):
        try: return datetime.strptime(str(s), fmt)
        except: pass
    return datetime.fromisoformat(str(s))

def cosine(a, b):
    if a is None or b is None or len(a) != len(b): return 0.0
    dot = sum(x*y for x,y in zip(a,b))
    na = math.sqrt(sum(x*x for x in a))
    nb = math.sqrt(sum(y*y for y in b))
    return dot/(na*nb) if na and nb else 0.0

def recency(created, now):
    dt_h = max(0.0, (now - created).total_seconds()/3600.0)
    return 2 ** (-dt_h / HALF_LIFE_HOURS)

def importance(poignancy):
    # GA-style importance is 1..10 → normalize to 0..1
    p = float(poignancy or 0.0)
    if p <= 1.0:         # many events are 1
        return p / 10.0  # treat as 0.1
    return min(1.0, p / 10.0)

def load_nodes():
    data = json.loads(NODES.read_text(encoding="utf-8"))
    nodes = list(data.values()) if isinstance(data, dict) else data
    for n in nodes:
        n["_t"] = parse_dt(n["created"])
        n["_k"] = (n.get("embedding_key") or n.get("description") or n.get("text") or "").strip()
    nodes.sort(key=lambda x: x["_t"])
    return nodes

def load_embs():
    e = json.loads(EMBS.read_text(encoding="utf-8"))
    return {k:v for k,v in e.items() if isinstance(v, list)}

def retrieve(nodes, embs, qvec, now=None):
    now = now or nodes[-1]["_t"]
    out = []
    for n in nodes:
        # expiration handling (optional)
        if n.get("expiration"):
            try:
                if parse_dt(n["expiration"]) <= now:
                    continue
            except:
                pass

        vec = embs.get(n["_k"])
        if vec is None:
            continue

        rel = (cosine(qvec, vec) + 1) / 2  # [-1,1]→[0,1]
        if rel < REL_MIN:
            continue

        r = recency(n["_t"], now)
        imp = importance(n.get("poignancy"))
        tw = TYPE_W.get(n.get("type"), 1.0)

        score = tw * (W_RECENCY*r + W_IMPORTANCE*imp + W_RELEVANCE*rel)
        out.append((score, r, imp, rel, tw, n))

    out.sort(key=lambda x: x[0], reverse=True)
    return out[:TOP_K]

if __name__ == "__main__":
    nodes = load_nodes()
    embs = load_embs()

    k = next(iter(embs))
    print("embeddings keys:", len(embs), "| dim:", len(embs[k]))

    model = SentenceTransformer(MODEL)

    query = "What is Klaus working on for his research?"
    qvec = model.encode([query], normalize_embeddings=True)[0].tolist()

    res = retrieve(nodes, embs, qvec)

    print("\nQUERY:", query)
    print("NOW:", nodes[-1]["_t"])
    print("\nTOP MEMORIES:")
    for score, r, imp, rel, tw, n in res:
        txt = (n.get("description") or n.get("text") or "")[:140]
        print(f"{score:.3f} tw={tw:.1f} r={r:.2f} imp={imp:.2f} rel={rel:.2f} [{n['type']}] {n['created']} :: {txt}")


c:\Users\srina\projects\Generative Agents and Memory-Based Retrieval\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


embeddings keys: 202 | dim: 384


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 837.65it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



QUERY: What is Klaus working on for his research?
NOW: 2023-02-14 00:02:10

TOP MEMORIES:
6.018 tw=2.0 r=0.65 imp=0.90 rel=0.88 [thought] 2023-02-13 09:15:10 :: Klaus Mueller is actively researching a topic
5.822 tw=2.0 r=0.83 imp=0.80 rel=0.86 [thought] 2023-02-13 17:47:20 :: Klaus Mueller is academically inclined
5.822 tw=2.0 r=0.83 imp=0.80 rel=0.86 [thought] 2023-02-13 17:47:20 :: Klaus Mueller is academically inclined
5.804 tw=2.0 r=0.77 imp=0.80 rel=0.86 [thought] 2023-02-13 15:12:10 :: Klaus Mueller is academically inclined
5.781 tw=2.0 r=0.70 imp=0.80 rel=0.86 [thought] 2023-02-13 11:35:20 :: Klaus Mueller is academically inclined
5.735 tw=2.0 r=0.77 imp=0.70 rel=0.88 [thought] 2023-02-13 15:12:10 :: Klaus Mueller is actively researching a topic
5.712 tw=2.0 r=0.70 imp=0.70 rel=0.88 [thought] 2023-02-13 11:35:20 :: Klaus Mueller is actively researching a topic
5.680 tw=2.0 r=0.89 imp=0.70 rel=0.86 [thought] 2023-02-13 20:07:40 :: Klaus Mueller is academically inclined
5.662 tw

In [2]:
queries = [
    "Valentine's Day party balloons snacks decorations",
    "Who did Klaus talk to about the Valentine's Day party?",
    "Who are Klaus's close friends?",
]

for query in queries:
    qvec = model.encode([query], normalize_embeddings=False)[0].tolist()
    res = retrieve(nodes, embs, qvec)

    print("\n" + "="*80)
    print("QUERY:", query)
    print("NOW:", nodes[-1]["_t"])
    print("\nTOP MEMORIES:")
    for score, r, imp, rel, tw, n in res:
        txt = (n.get("description") or n.get("text") or "")[:140]
        print(f"{score:.3f} [{n.get('type')}] {n.get('created')} :: {txt}")


QUERY: Valentine's Day party balloons snacks decorations
NOW: 2023-02-14 00:02:10

TOP MEMORIES:
4.744 [thought] 2023-02-13 11:24:50 :: Klaus Mueller Klaus found the invitation to the Valentine's Day party interesting.
4.695 [thought] 2023-02-13 17:34:50 :: For Klaus Mueller's planning: needs to remember to bring some red and pink balloons and homemade cookies and chips for the Valentine's Day p
4.601 [thought] 2023-02-13 17:47:20 :: Klaus Mueller is invited to Isabella Rodriguez's Valentine's Day party at Hobbs Cafe
4.559 [thought] 2023-02-13 11:24:50 :: For Klaus Mueller's planning: needs to remember to attend Isabella Rodriguez's Valentine's Day party at Hobbs Cafe on February 14th, 2023 fr
4.406 [thought] 2023-02-13 12:35:00 :: Klaus Mueller is invited to Isabella Rodriguez's Valentine's Day party at Hobbs Cafe
4.406 [thought] 2023-02-13 12:35:00 :: Klaus Mueller is invited to Isabella Rodriguez's Valentine's Day party at Hobbs Cafe
4.400 [thought] 2023-02-13 11:35:20 :: Klaus Mue